# Axon SFT — Diagnostic Notebook (cell 11 bottleneck probe)

**Goal:** run SFT for a handful of steps with CUDA-synced timing probes to find out which section of the training step is eating the time.

What this notebook does, and nothing else:

1. Pulls latest code from GitHub
2. Mounts Drive and links datasets
3. Builds an SFT config with `batch_size=4`, `num_epochs=1`, `profile_first_n_steps=5`
4. Slices the dataset to 32 samples so "1 epoch" = ~8 training steps
5. Runs SFT and prints per-section timings for every step

Expected runtime: ~1–2 minutes if the fixes work. If any single step takes longer than 30 seconds, stop the cell and paste the timing lines from the output — that tells us exactly where the remaining seconds are.

**Do not edit anything.** Just run top-to-bottom on a GPU runtime (`Runtime > Change runtime type > GPU`).

## 1. Pull latest code + install deps

In [ ]:
# Pull latest from GitHub
import os

if os.path.exists("Axon"):
    %cd Axon
    !git pull origin main
else:
    !git clone https://github.com/tylermark/Axon.git
    %cd Axon

!pip install -q pydantic pymupdf scipy networkx wandb timm 2>/dev/null

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    gpu_mem_gb = (getattr(props, "total_memory", None) or getattr(props, "total_mem", 0)) / 1e9
    print(f"GPU:     {torch.cuda.get_device_name(0)} ({gpu_mem_gb:.1f} GB)")

## 2. Mount Drive + link datasets

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/axon"
CKPT_DIR = f"{DRIVE_ROOT}/checkpoints"
DATA_DIR = f"{DRIVE_ROOT}/datasets"

import os
for d in [CKPT_DIR, DATA_DIR]:
    os.makedirs(d, exist_ok=True)

# Link archcad400k into the working directory
from pathlib import Path
local_data = Path("datasets")
local_data.mkdir(exist_ok=True)

src = Path(DATA_DIR) / "archcad400k"
target = local_data / "archcad400k"
if src.exists() and not target.exists():
    os.symlink(str(src), str(target))

assert src.exists(), f"archcad400k not found at {src} — upload it first"
print(f"Dataset linked: {target} -> {src}")

## 3. Build tiny SFT config + tiny dataset slice

In [ ]:
import torch
from src.training.sft import SFTTrainer, SFTConfig

# Diagnostic config: short, small, verbose.
sft_config = SFTConfig(
    num_epochs=1,
    batch_size=4,                  # small — ramp later only if step time is healthy
    learning_rate=5e-5,
    checkpoint_dir="/tmp/sft_diag",  # throwaway dir — we don't care about checkpoints
    device="cuda" if torch.cuda.is_available() else "cpu",
    wandb_enabled=False,           # no W&B noise for a diagnostic run
    eval_every_n_epochs=999,       # skip eval
    save_every_n_epochs=999,       # skip checkpoint saves
    num_workers=0,
    profile_first_n_steps=9999,    # profile every step of this tiny run
)

print(f"Profile first {sft_config.profile_first_n_steps} steps")
print(f"Batch size: {sft_config.batch_size}")
print(f"Device:     {sft_config.device}")

## 4. Build models + tiny dataset

In [ ]:
import torch
from src.pipeline.config import AxonConfig
from src.tokenizer.cross_attention import Tokenizer
from src.diffusion.reverse import GraphDiffusionModel
from src.constraints.sat_solver import ConstraintSolver
from src.training.sft_data import SFTDatasetAdapter
from src.training.data_engine import ArchCAD400KDataset

# ── Models ──
config = AxonConfig()
tokenizer_model = Tokenizer(config=config.tokenizer)
diffusion_model = GraphDiffusionModel(config=config.diffusion)
constraint_module = ConstraintSolver(config=config.constraints)

param_count = sum(
    sum(p.numel() for p in m.parameters())
    for m in [tokenizer_model, diffusion_model, constraint_module]
)
print(f"Total params: {param_count:,}")

# ── Dataset ──
base_dataset = ArchCAD400KDataset(data_root="datasets/")
full_dataset = SFTDatasetAdapter(base_dataset, max_nodes=512)
print(f"Full dataset: {len(full_dataset)} samples")

# DIAGNOSTIC: slice to 32 samples so 1 epoch = 8 steps at batch=4.
tiny_dataset = torch.utils.data.Subset(full_dataset, range(min(32, len(full_dataset))))
print(f"Tiny slice:   {len(tiny_dataset)} samples  →  ~{len(tiny_dataset) // sft_config.batch_size} steps/epoch")

## 5. Run SFT with timing probes

Watch the output. You should see one line per training step that looks like:

```
step 0 timing: tokenizer_fwd=12ms diffusion_fwd=45ms edges_from_adj=3ms constraint_fwd=80ms backward=120ms optimizer=8ms
```

Any section stuck in the seconds is the next thing to chase. If a step hangs for more than ~30 seconds, interrupt the cell and paste whatever timing lines appeared.

In [ ]:
import logging
# Make sure INFO-level logs from SFTTrainer show up in the notebook.
logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)

sft_trainer = SFTTrainer(
    tokenizer_model=tokenizer_model,
    diffusion_model=diffusion_model,
    constraint_module=constraint_module,
    dataset=tiny_dataset,
    eval_dataset=None,
    config=sft_config,
)

sft_trainer.train()
print("\nDiagnostic run complete.")